# Model Performance Report

This notebook is the display surface for the final model-performance table. It uses reusable functions from `src/`, directly displays model performance evidence, without writing a CSV export.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.audit import add_visit_time, modelling_pair_count_table
from src.data.trackfa import feature_catalog
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.clinical_validity import clinical_validity
from src.eval.intervals import adjacent_pair_interval_effect_summary, interval_effect_summary, pooled_adjacent_pair_effect_summary
from src.eval.single_feature import single_feature_interval_baselines
from src.eval.stability import selected_feature_jaccard
from src.models.srm_global import srm_global_loocv, srm_global_repeated_group_cv
from src.reporting.model_performance import (
    append_log_model_summaries,
    assemble_performance_rows,
    best_model_rows_from_logs,
    validation_test_gap_table,
)
from src.reporting.fold_comparison import fold_train_test_clinical_benchmark_table

set_global_seeds(DEFAULT_CONFIG.random_state)
RANDOM_SEED = DEFAULT_CONFIG.random_state
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 300

long_path = REPO_ROOT / "data" / "processed" / "trackfa_long.csv"
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")

long_df = add_visit_time(pd.read_csv(long_path), visit_col="visit")
pairs_df = pd.read_csv(pairs_path)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
pair_long_df = trackfa_pairs_to_long(pairs_df)
feature_groups = infer_trackfa_feature_groups(pairs_df)
cat = feature_catalog(long_df, pairs_df)
imaging_cols = [c for c in feature_groups.all_neuroimaging if c in long_df.columns]
if not imaging_cols:
    imaging_cols = cat["long_imaging_columns"]

print(f"Loaded {long_path.name}: {long_df.shape[0]} rows, {long_df['subject_id'].nunique()} subjects")
print(f"Loaded {pairs_path.name}: {pairs_df.shape[0]} rows")
print(f"MRI features available for reporting: {len(imaging_cols)}")


In [ ]:
# Fold-level clinical train/test benchmark for supervisor review.
fold_train_test_clinical_benchmarks = fold_train_test_clinical_benchmark_table(
    pair_long_df,
    pairs_df,
    [c for c in feature_groups.all_neuroimaging if c in pair_long_df.columns],
    subject_col="pair_id",
    visit_col="visit",
    split_group_col="subject",
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    clinical_scales=("FARS", "SARA"),
    pair_types=("V1V2", "V2V3"),
)
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
fold_train_test_clinical_benchmarks.to_csv(
    RESULTS_DIR / "model_performance_fold_train_test_clinical_benchmarks.csv",
    index=False,
)
clinical_display_cols = [
    "fold",
    "clinical_scale",
    "train_n_subjects",
    "test_n_subjects",
    "clinical_train_n_pairs",
    "clinical_test_n_pairs",
    "clinical_train_d",
    "clinical_test_d",
    "clinical_train_minus_test_d",
]
print("Model performance fold-level train/test clinical benchmarks")
display(fold_train_test_clinical_benchmarks[clinical_display_cols])


## Model Performance Table

The SRM Global Linear model is recomputed here on the subject-level longitudinal table for the primary and secondary intervals. Other model families are appended from existing optimization logs using the same required question/metric schema; unavailable metrics remain explicitly marked as unavailable rather than inferred.


In [ ]:
pair_imaging_cols = [c for c in feature_groups.all_neuroimaging if c in pair_long_df.columns]
annual_pooled_result = srm_global_loocv(
    pair_long_df,
    pair_imaging_cols,
    subject_col="pair_id",
    visit_col="visit",
    selection_method="none",
    k=8,
    ridge=1e-6,
    covariance_shrinkage=0.45,
    z_clip=None,
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    compute_ci=False,
    split_group_col="subject",
    start_visit=1,
    end_visit=2,
)
annual_interval_summary = adjacent_pair_interval_effect_summary(
    annual_pooled_result["oof_df"],
    pair_col="pair_id",
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)

# Reconstruct V1->V3 from annual-pair OOF scores for subjects with both V1V2 and V2V3 rows.
oof = annual_pooled_result["oof_df"].copy()
parsed = oof["pair_id"].astype(str).str.extract(r"(?P<subject>.+)_(?P<pair_type>V1V2|V2V3)$")
oof = oof.assign(subject_id=parsed["subject"], pair_type=parsed["pair_type"])
v1_scores = (
    oof.loc[oof["pair_type"].eq("V1V2") & oof["visit"].eq(1), ["subject_id", "score"]]
    .rename(columns={"score": "score_v1"})
)
v3_scores = (
    oof.loc[oof["pair_type"].eq("V2V3") & oof["visit"].eq(2), ["subject_id", "score"]]
    .rename(columns={"score": "score_v3"})
)
v13_pairs = v1_scores.merge(v3_scores, on="subject_id", how="inner")
v13_long = pd.concat([
    v13_pairs[["subject_id", "score_v1"]].rename(columns={"score_v1": "score"}).assign(visit=1, time_years=0.0),
    v13_pairs[["subject_id", "score_v3"]].rename(columns={"score_v3": "score"}).assign(visit=3, time_years=2.0),
], ignore_index=True)
v13_summary = interval_effect_summary(
    v13_long,
    subject_col="subject_id",
    visit_col="visit",
    score_col="score",
    time_col="time_years",
    intervals=[(1, 3, "V1->V3", False)],
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)

pooled_annual_summary = pooled_adjacent_pair_effect_summary(
    annual_pooled_result["oof_df"],
    pair_col="pair_id",
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
composite_intervals = pd.concat([
    annual_interval_summary,
    v13_summary,
    pooled_annual_summary,
], ignore_index=True)
print("Composite interval performance from annual pair-table OOF predictions")
display(composite_intervals)
print("Counts are tied to trackfa_pairs_drop3poms.csv: N12=108, N23=99, N13=90, N123=90. Pooled annual d_z uses pair OOF predictions with split_group_col='subject'.")

srm_interval_results = {
    "V1->V2": annual_pooled_result,
    "V2->V3": annual_pooled_result,
    "V1->V3": {"oof_df": v13_long, "selected_features_by_fold": annual_pooled_result["selected_features_by_fold"]},
}


In [ ]:
single_feature_intervals = single_feature_interval_baselines(
    long_df,
    imaging_cols,
    subject_col="subject_id",
    visit_col="visit",
    time_col="time_years",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
print("Strongest individual MRI features by interval")
display(single_feature_intervals.groupby("interval", group_keys=False).head(5))

clinical_vars = [c for c in ["FARS", "SARA", "mfars_total", "sara_total"] if c in long_df.columns]
clinical_interval_parts = []
for clinical_col in clinical_vars:
    clinical_interval_parts.append(
        interval_effect_summary(
            long_df.dropna(subset=[clinical_col]),
            subject_col="subject_id",
            visit_col="visit",
            score_col=clinical_col,
            time_col="time_years",
            n_boot=N_BOOT,
            seed=RANDOM_SEED,
        ).assign(feature=clinical_col, kind="clinical_scale")
    )
clinical_intervals = pd.concat(clinical_interval_parts, ignore_index=True) if clinical_interval_parts else pd.DataFrame()
print("Clinical-scale benchmarks")
display(clinical_intervals)


In [ ]:
def clinical_annual_effects_from_pairs(pairs: pd.DataFrame, delta_cols: list[str]) -> pd.DataFrame:
    """Recalculate annual clinical benchmarks directly from paired annual deltas."""
    interval = pairs["patient_id"].astype(str).str.extract(r"_(V\dV\d)$")[0]
    rows = []
    interval_specs = [
        ("V1->V2", interval.eq("V1V2")),
        ("V2->V3", interval.eq("V2V3")),
        ("pooled annual", interval.isin(["V1V2", "V2V3"])),
    ]
    for col in delta_cols:
        clinical_name = col.removeprefix("delta_")
        for label, mask in interval_specs:
            values = pd.to_numeric(pairs.loc[mask, col], errors="coerce").dropna()
            sd = values.std(ddof=1)
            rows.append({
                "clinical_score": clinical_name,
                "interval": label,
                "n_pairs": int(values.shape[0]),
                "mean_delta": values.mean(),
                "sd_delta": sd,
                "d_z": values.mean() / sd if sd and not np.isnan(sd) else np.nan,
                "p_delta_gt_0": (values > 0).mean() if values.shape[0] else np.nan,
            })
    return pd.DataFrame(rows)

clinical_delta_cols = [
    c for c in ["delta_mfars_total", "delta_sara_total", "delta_adl_total"] if c in pairs_df.columns
]
clinical_annual_pair_benchmarks = clinical_annual_effects_from_pairs(pairs_df, clinical_delta_cols)
print("Clinical annual benchmarks recalculated directly from trackfa_pairs_drop3poms.csv")
display(clinical_annual_pair_benchmarks)

clinical_pooled_confirmation = (
    clinical_annual_pair_benchmarks
    .query("interval == 'pooled annual'")
    .loc[:, ["clinical_score", "n_pairs", "mean_delta", "sd_delta", "d_z", "p_delta_gt_0"]]
    .sort_values("clinical_score")
    .reset_index(drop=True)
)
print("Pooled annual clinical d_z confirmation")
display(clinical_pooled_confirmation)

for score in ["mfars_total", "sara_total"]:
    match = clinical_pooled_confirmation.loc[clinical_pooled_confirmation["clinical_score"].eq(score)]
    if not match.empty:
        row = match.iloc[0]
        print(
            f"Confirmed {score} pooled annual d_z = {row['d_z']:.3f} "
            f"(N={int(row['n_pairs'])}, P(delta>0)={row['p_delta_gt_0']:.3f})."
        )


In [ ]:
primary_oof = srm_interval_results["V1->V3"]["oof_df"]
clinical_validity_table = clinical_validity(
    primary_oof,
    long_df,
    subject_col="subject_id",
    visit_col="visit",
    score_col="score",
    clinical_variables=clinical_vars,
    start_visit="V1",
    end_visit="V3",
) if clinical_vars else pd.DataFrame()
print("Clinical validation of locked OOF composite scores")
display(clinical_validity_table)

stability_table = pd.DataFrame([
    {
        "model": "SRM Global Linear",
        "mean_jaccard": selected_feature_jaccard(
            [fs for result in srm_interval_results.values() for fs in result["selected_features_by_fold"]]
        )["mean_jaccard"],
        "sign_stability": np.nan,
        "score_ranking_stability": np.nan,
    }
])
print("Feature robustness diagnostics")
display(stability_table)


In [ ]:
log_models = best_model_rows_from_logs([
    REPO_ROOT / "results" / "srm_composite_optimization_log.csv",
    REPO_ROOT / "results" / "comparator_optimization_log.csv",
    REPO_ROOT / "results" / "progression_dl_optimization_log.csv",
])

srm_log_row = None
if not log_models.empty and "model" in log_models.columns:
    srm_hits = log_models[log_models["model"].astype(str).str.contains("SRM Global Linear", regex=False, na=False)]
    if not srm_hits.empty:
        srm_log_row = srm_hits.iloc[0]

validation_test_review = validation_test_gap_table(
    "SRM Global Linear",
    test_intervals=composite_intervals,
    log_row=srm_log_row,
)
print("Validation vs held-out/test overfit check")
display(validation_test_review)

performance = assemble_performance_rows(
    "SRM Global Linear",
    composite_intervals=composite_intervals,
    clinical_intervals=clinical_intervals,
    single_feature_intervals=single_feature_intervals,
    clinical_validity=clinical_validity_table,
    stability=stability_table,
    validation_test=validation_test_review,
    cv_mode=f"subject-level grouped {CV_N_SPLITS}-fold",
    source="model_performance.ipynb",
)

performance = append_log_model_summaries(performance, log_models)
print("Model evaluation table with validation/test columns")
display(performance)
